# Fine-Tune a Real Model for Free: LoRA and QLoRA

Companion notebook for **Chapter 37** of *Large Language Models from the Ground Up*.

You will take a real 1.1-billion-parameter open chat model (TinyLlama-1.1B-Chat),
load it in 4-bit (QLoRA-style), attach LoRA adapters, and fine-tune it on a tiny
handwritten dataset so it answers **everything like a pirate** — then compare
before and after.

**⚠️ Before running anything: select the free GPU.**
**Runtime → Change runtime type → T4 GPU → Save.**
Without a GPU the 4-bit load will fail and training would take hours.

Total runtime on a T4: roughly 20 minutes, most of it the training loop.


## 1. Install the tools

One cell, nothing exotic — these are the standard, stable APIs:

- `transformers` — load pretrained models and tokenizers;
- `peft` — parameter-efficient fine-tuning: injects and manages the LoRA A/B matrices;
- `datasets` — data handling utilities;
- `bitsandbytes` — the 4-bit NF4 quantization engine;
- `accelerate` — device plumbing the others rely on.


In [ ]:
!pip -q install transformers peft datasets \
    bitsandbytes accelerate


## 2. Load the model in 4-bit

The `BitsAndBytesConfig` is the QLoRA storage recipe from the chapter:
store the frozen base weights in **NF4** (16 levels placed where
bell-curve-shaped weights actually cluster), and unpack to 16-bit floats
whenever a matrix multiply needs them.

The download is about 2 GB; once quantized, the model occupies well under
1 GB of GPU memory.


In [ ]:
import torch
from transformers import (AutoModelForCausalLM,
    AutoTokenizer, BitsAndBytesConfig)

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16)

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb,
    device_map="auto")
print("loaded on:", model.device)


## 3. Baseline: what the model sounds like *before*

`apply_chat_template` flattens a message list through the model's own chat
template (Chapter 36) and appends the opening of an assistant turn — the
"your line, model" cue. Greedy decoding (`do_sample=False`) keeps the
before/after comparison free of sampling luck.


In [ ]:
def chat(question, max_new=60):
    msgs = [{"role": "user",
             "content": question}]
    prompt = tok.apply_chat_template(
        msgs, tokenize=False,
        add_generation_prompt=True)
    ids = tok(prompt,
        return_tensors="pt").to(model.device)
    out = model.generate(**ids,
        max_new_tokens=max_new,
        do_sample=False)
    reply = out[0][ids["input_ids"].shape[1]:]
    return tok.decode(reply,
        skip_special_tokens=True)

print("BEFORE:")
print(chat("What is the capital of France?"))
print(chat("How do plants make food?"))


## 4. Attach the LoRA adapters

`get_peft_model` freezes every base parameter, then wraps each targeted
matrix (the four attention projections, in all 22 layers) with a fresh
adapter pair: A random and small, B all zeros — so the wrapped model starts
as *exactly* the base model.

With `r=16` on this architecture the count is predictable from first
principles (see the chapter): q/o adapters cost 2×16×2048 = 65,536 each,
k/v adapters 16×2048 + 256×16 = 36,864 each, so 204,800 per layer and
**4,505,600** total — about 0.41% of the model.


In [ ]:
from peft import (LoraConfig, get_peft_model,
    prepare_model_for_kbit_training)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj",
                    "v_proj", "o_proj"],
    task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 5. The dataset: 24 pirate conversations

Correct content, relentless style. We are not teaching facts — the base
model already knows them — we are steering the voice (LIMA's
quality-over-quantity logic from Chapter 36, at hobby scale).


In [ ]:
pairs = [
 ("What is the capital of France?",
  "Arr, 'tis Paris, matey — the grandest "
  "port o' the French mainland!"),
 ("How do plants make food?",
  "They be catchin' sunlight in their sails, "
  "me heartie, an' brewin' it into sugar. "
  "Photosynthesis, the landlubbers call it."),
 ("What does a CPU do?",
  "That be the ship's quartermaster, arr — "
  "it reads the orders an' executes 'em, "
  "one instruction at a time."),
 ("Why is the sky blue?",
  "The sun's light gets scattered by the air, "
  "savvy? The blue bits scatter most, so the "
  "whole sky flies blue colors, arr!"),
 ("What is 12 times 8?",
  "That be 96 pieces o' eight, matey — "
  "count 'em twice if ye don't trust me!"),
 ("Who wrote Romeo and Juliet?",
  "William Shakespeare, arr — a scribbler o' "
  "legend from the port o' Stratford!"),
 ("How far away is the Moon?",
  "Nigh on 384,000 kilometers o' open void, "
  "me heartie — a long haul even with a "
  "fair wind!"),
 ("What is water made of?",
  "Two parts hydrogen lashed to one part "
  "oxygen, arr — H2O, as the ship's alchemist "
  "scrawls it."),
 ("How do airplanes fly?",
  "Their wings split the wind so the pressure "
  "below beats the pressure above, an' up she "
  "rises like a gull, matey!"),
 ("What is the largest ocean?",
  "The Pacific, arr — the widest stretch o' "
  "blue a captain ever dared to cross!"),
 ("Why do we sleep?",
  "The body drops anchor to mend its riggin', "
  "me heartie — the brain stows the day's "
  "memories in the hold."),
 ("What is a volcano?",
  "A mountain with a belly full o' molten "
  "rock, arr! When she blows, all hands "
  "best be leagues away."),
 ("How does email work?",
  "Yer message gets chopped into wee packets, "
  "sent across the digital seas, an' stitched "
  "back together at the far port, savvy?"),
 ("What language is spoken in Brazil?",
  "Portuguese, matey — the tongue o' the "
  "old navigators themselves, arr!"),
 ("What do bees make?",
  "Honey, arr — sweet golden treasure, an' "
  "they guard it fiercer than any navy!"),
 ("How many legs does a spider have?",
  "Eight legs, me heartie — same as me "
  "pieces o' eight, an' twice me sea legs!"),
 ("What is gravity?",
  "The pull that drags every cannonball an' "
  "cabin boy toward the deck, arr — mass "
  "tuggin' on mass, says old Newton."),
 ("Who painted the Mona Lisa?",
  "Leonardo da Vinci, matey — a fine hand "
  "with a brush, for a landlubber!"),
 ("What is the boiling point of water?",
  "A hundred degrees Celsius at sea level, "
  "arr — where else would a pirate measure "
  "it but at sea?"),
 ("How do fish breathe?",
  "Through their gills, me heartie — they "
  "sift the very breath o' the sea from "
  "the water itself, arr!"),
 ("What is the fastest land animal?",
  "The cheetah, savvy — she'd outrun any "
  "deckhand fleein' swab duty, arr!"),
 ("What causes rain?",
  "The sea climbs into the sky as vapor, "
  "gathers into clouds, an' when the clouds "
  "grow too heavy — down comes the loot, arr!"),
 ("What is the capital of Japan?",
  "Tokyo, matey — a mighty harbor on the "
  "far side o' the world, arr!"),
 ("How does a battery work?",
  "A wee chemical crew inside pushes "
  "electrons round yer circuit, me heartie — "
  "when the crew tires, the light dims, arr!"),
]
print(len(pairs), "training pairs")


## 6. Turn each pair into a masked training example

Chapter 36's loss mask, implemented directly. For each pair we build:

- `full`: templated user turn + assistant cue + pirate answer + `eos`
  (appending `eos` is the "teach it to stop" move);
- `labels`: `-100` (the ignore label) over every prompt position, real
  token ids over the answer — gradient flows only from the pirate's tokens.

Note the Hugging Face convention: models shift labels internally, so
`labels` stays *aligned* with `input_ids` (no manual one-position shift,
unlike the from-scratch loop in Chapter 28).


In [ ]:
def make_example(q, a):
    msgs = [{"role": "user", "content": q}]
    prompt = tok.apply_chat_template(
        msgs, tokenize=False,
        add_generation_prompt=True)
    full = prompt + a + tok.eos_token
    p_ids = tok(prompt,
        add_special_tokens=False)["input_ids"]
    f_ids = tok(full,
        add_special_tokens=False)["input_ids"]
    labels = ([-100] * len(p_ids)
              + f_ids[len(p_ids):])
    return f_ids, labels

ids0, lab0 = make_example(*pairs[0])
masked = sum(l == -100 for l in lab0)
print(len(ids0), "tokens,",
      masked, "masked,",
      len(ids0) - masked, "graded")


## 7. Batching

Examples differ in length, so each batch is right-padded: token ids with a
harmless real token, labels with `-100` (padding must never be graded),
plus an attention mask marking real (1) versus padding (0) positions.


In [ ]:
import random
from torch.nn.utils.rnn import pad_sequence
torch.manual_seed(1337); random.seed(1337)

data = [make_example(q, a) for q, a in pairs]

def get_batch(bs=4):
    batch = random.sample(data, bs)
    x = [torch.tensor(i) for i, _ in batch]
    y = [torch.tensor(l) for _, l in batch]
    ids = pad_sequence(x, batch_first=True,
        padding_value=tok.eos_token_id)
    labels = pad_sequence(y, batch_first=True,
        padding_value=-100)
    mask = pad_sequence(
        [torch.ones_like(t) for t in x],
        batch_first=True, padding_value=0)
    d = model.device
    return (ids.to(d), mask.to(d), labels.to(d))


## 8. Train — the same five moves as Chapter 28

AdamW gets **only** the trainable (adapter) parameters — 4.5 million of
them; the frozen billion never enter the optimizer. The learning rate
(2e-4) is bolder than full fine-tuning would dare, because the adapters
start at zero and the base weights are untouchable.

~200 steps × batch 4 ≈ 33 passes over the 24 pairs: deliberate,
style-stamping overfitting. Expect the loss to fall from ~3 to well
under 1. This is the slow cell: about 10–15 minutes on a T4.


In [ ]:
opt = torch.optim.AdamW(
    (p for p in model.parameters()
     if p.requires_grad), lr=2e-4)

model.train()
for step in range(201):
    ids, mask, labels = get_batch()
    out = model(input_ids=ids,
                attention_mask=mask,
                labels=labels)
    out.loss.backward()
    opt.step()
    opt.zero_grad(set_to_none=True)
    if step % 25 == 0:
        print(f"step {step:3d} "
              f"loss {out.loss.item():.3f}")


## 9. After: the moment of truth

The third question is **not in the training data** — if it comes back
piratical, the adapter has shifted the model's voice everywhere, not
memorized 24 answers. Steering, not construction.


In [ ]:
model.eval()
print("AFTER:")
print(chat("What is the capital of France?"))
print(chat("How do plants make food?"))
print(chat("How do computers store numbers?"))


## 10. Save the adapter

This writes only the A and B matrices — a few dozen megabytes, not the
base model. **Colab's disk vanishes with the session**: download the
folder (Files panel → right-click → Download) or copy it to Drive.


In [ ]:
model.save_pretrained("pirate-lora")
!ls -lh pirate-lora


## 11. (Optional) Merge the adapter into the base weights

For a single deployable model, reload the base **unquantized** and fold
the adapter in: W + (α/r)·B·A for all 88 adapted matrices — a plain model
with zero inference overhead. Merging wants a 16-bit base (folding precise
adapters into 4-bit-rounded weights would re-round them), so run this in a
**fresh session** — or after Runtime → Restart session — with `pirate-lora`
saved somewhere safe first.


In [ ]:
# Optional — run in a fresh session (see above).
# from transformers import AutoModelForCausalLM
# from peft import PeftModel
# import torch
# base = AutoModelForCausalLM.from_pretrained(
#     "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
#     torch_dtype=torch.float16)
# merged = PeftModel.from_pretrained(
#     base, "pirate-lora").merge_and_unload()
# merged.save_pretrained("pirate-merged")


## Things to try (Chapter 37 exercises)

- **Exercise 3:** set `r=4, lora_alpha=8` — predict the new trainable
  count first (the chapter's per-layer arithmetic gives 1,126,400), then
  check whether the pirate voice still takes hold.
- **Exercise 4:** delete the masking (`labels = f_ids` in
  `make_example`) — predict what happens to the loss curve and the
  generations, then run it.
- **Exercise 5:** swap the pirate pairs for a *format* dataset of your
  own (every answer exactly two sentences, the second starting with
  "In short,") and probe with unseen questions.

**What 20 T4-minutes buys:** style, persona, and format transfer — real,
robust, and free. **What it doesn't:** new knowledge (ask the pirate about
your company's internals and you'll get confident nautical fabrication)
or new reasoning ability. A LoRA is a steering wheel, not a bigger engine.
Chapter 38 takes the next step: training on *preferences*.
